---
## 0. Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q ultralytics segment-anything scikit-learn tqdm

In [ ]:
# Download SAM ViT-B checkpoint (used by MedSAM)
import os, subprocess

ckpt_path = "/kaggle/working/medsam_vit_b.pth"
if not os.path.exists(ckpt_path):
    subprocess.run(
        ["wget", "-q",
         "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
         "-O", ckpt_path],
        check=True,
    )
    print(f"Downloaded SAM ViT-B checkpoint to {ckpt_path}")
else:
    print(f"SAM ViT-B checkpoint already exists at {ckpt_path}")

In [ ]:
# Copy src from Kaggle input (read-only) to working directory,
# then write Kaggle-specific config files so all path references are correct.
import shutil
from pathlib import Path

SRC_INPUT   = Path("/kaggle/input/datasets/nguyenquynghia/src-training/src")
SRC_WORKING = Path("/kaggle/working/src")

if SRC_WORKING.exists():
    shutil.rmtree(SRC_WORKING)
shutil.copytree(SRC_INPUT, SRC_WORKING)
print(f"Copied source to {SRC_WORKING}")

# ── Kaggle-specific yolo/config.py ──────────────────────────────────────
YOLO_CONFIG = '''\
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────
ISIC_BASE = Path("/kaggle/input/datasets/nguyenquynghia/isic-2018")
WORKING_DIR = Path("/kaggle/working")

TASK1_2_INPUT = ISIC_BASE / "ISIC2018_Task1-2_Training_Input"
TASK1_GT      = ISIC_BASE / "ISIC2018_Task1_Training_GroundTruth"

# Prepared dataset
DATASET_DIR = WORKING_DIR / "dataset"
YOLO_DIR    = DATASET_DIR / "yolo"

# Output
OUTPUT_DIR  = WORKING_DIR / "outputs"
YOLO_OUTPUT = OUTPUT_DIR  / "yolo"

# ── Dataset ──────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

# ── YOLOv8 ─────────────────────────────────────────────────────────────────
YOLO_MODEL    = "yolov8n.pt"
YOLO_EPOCHS   = 100
YOLO_BATCH    = 16
YOLO_IMG_SIZE = 640
'''

(SRC_WORKING / "yolo" / "config.py").write_text(YOLO_CONFIG)
print("Wrote yolo/config.py")

# ── Kaggle-specific medsam/config.py ────────────────────────────────────
MEDSAM_CONFIG = '''\
from pathlib import Path
import torch

# ── Paths ──────────────────────────────────────────────────────────────
ISIC_BASE   = Path("/kaggle/input/datasets/nguyenquynghia/isic-2018")
WORKING_DIR = Path("/kaggle/working")

TASK1_2_INPUT = ISIC_BASE / "ISIC2018_Task1-2_Training_Input"
TASK1_GT      = ISIC_BASE / "ISIC2018_Task1_Training_GroundTruth"
TASK2_GT      = ISIC_BASE / "ISIC2018_Task2_Training_GroundTruth_v3"

# Prepared dataset directories
DATASET_DIR   = WORKING_DIR / "dataset"
MEDSAM_DIR    = DATASET_DIR / "medsam"
ATTR_DIR      = DATASET_DIR / "attributes"

# Output directories
OUTPUT_DIR    = WORKING_DIR / "outputs"
MEDSAM_OUTPUT = OUTPUT_DIR  / "medsam"
ATTR_OUTPUT   = OUTPUT_DIR  / "attributes"

# ── Dataset ──────────────────────────────────────────────────────────────
ATTRIBUTES     = ["globules", "milia_like_cyst", "negative_network", "pigment_network", "streaks"]
NUM_ATTRIBUTES = len(ATTRIBUTES)
TRAIN_RATIO    = 0.8
RANDOM_SEED    = 42

# ── Preprocessing ──────────────────────────────────────────────────────────
MEDSAM_IMG_SIZE = 1024
ATTR_IMG_SIZE   = 256

# ── MedSAM ───────────────────────────────────────────────────────────────
MEDSAM_CHECKPOINT    = "/kaggle/working/medsam_vit_b.pth"
SAM_MODEL_TYPE       = "vit_b"
MEDSAM_LR            = 1e-4
MEDSAM_WEIGHT_DECAY  = 0.01
MEDSAM_EPOCHS        = 50
MEDSAM_BATCH         = 4
BBOX_SHIFT           = 20
CROP_MARGIN          = 50

# ── Attribute Segmentation ─────────────────────────────────────────────────────────
ATTR_LR           = 1e-4
ATTR_WEIGHT_DECAY = 0.01
ATTR_EPOCHS       = 80
ATTR_BATCH        = 4

# ── Device ───────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
'''

(SRC_WORKING / "medsam" / "config.py").write_text(MEDSAM_CONFIG)
print("Wrote medsam/config.py")

In [ ]:
# Add working src to Python path
import sys

src_path = "/kaggle/working/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f"Python path includes: {src_path}")

---
## Stage 1 — YOLOv8 Lesion Detection

Prepare YOLO-format dataset from ISIC segmentation masks, then train YOLOv8 for bounding-box detection.

In [ ]:
# 1.1 Prepare YOLO dataset
from yolo.prepare_data import get_image_ids, split_ids, prepare_yolo

image_ids = get_image_ids()
print(f"Found {len(image_ids)} images with segmentation masks")
train_ids, val_ids = split_ids(image_ids)
print(f"Split: {len(train_ids)} train, {len(val_ids)} val")
prepare_yolo(train_ids, val_ids)

In [ ]:
# 1.2 Train YOLOv8
from yolo.train import train as train_yolo

model_yolo = train_yolo(epochs=100, batch=16, img_size=640)

In [ ]:
# 1.3 Validate YOLOv8
from yolo.train import validate as validate_yolo

validate_yolo()

In [ ]:
# 1.4 Inference on a single image and visualize (GT vs Prediction)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
from pathlib import Path
from yolo.config import YOLO_OUTPUT, YOLO_DIR
from ultralytics import YOLO
weights_path = str(YOLO_OUTPUT / "lesion_detect" / "weights" / "best.pt")
model = YOLO(weights_path)

# Pick one validation image
val_dir = YOLO_DIR / "images" / "val"
sample_img = sorted(val_dir.glob("*.jpg"))[0]
print(f"Running inference on: {sample_img.name}")

# Load image
img = np.array(Image.open(sample_img).convert("RGB"))
H, W = img.shape[:2]

# Load ground-truth label (YOLO format: class cx cy w h, normalized)
label_path = YOLO_DIR / "labels" / "val" / sample_img.with_suffix(".txt").name
gt_boxes = []
if label_path.exists():
    for line in label_path.read_text().strip().splitlines():
        parts = list(map(float, line.split()))
        _, cx, cy, bw, bh = parts
        x1 = (cx - bw / 2) * W
        y1 = (cy - bh / 2) * H
        x2 = (cx + bw / 2) * W
        y2 = (cy + bh / 2) * H
        gt_boxes.append((x1, y1, x2, y2))

# Run YOLO prediction
results = model.predict(source=str(sample_img), imgsz=640, conf=0.25, verbose=False)
result = results[0]
pred_boxes = result.boxes.xyxy.cpu().numpy()  # (N, 4)
pred_confs = result.boxes.conf.cpu().numpy()

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, title in zip(axes, ["Ground Truth", "Prediction"]):
    ax.imshow(img)
    ax.set_title(f"{sample_img.name}\n{title}", fontsize=12)
    ax.axis("off")

# Draw GT boxes (green)
for (x1, y1, x2, y2) in gt_boxes:
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="lime", facecolor="none",
    )
    axes[0].add_patch(rect)
    axes[0].text(x1, y1 - 4, "lesion (GT)", color="lime", fontsize=9,
                 bbox=dict(facecolor="black", alpha=0.4, pad=1))

# Draw predicted boxes (red)
for (x1, y1, x2, y2), conf in zip(pred_boxes, pred_confs):
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="red", facecolor="none",
    )
    axes[1].add_patch(rect)
    axes[1].text(x1, y1 - 4, f"lesion {conf:.2f}", color="red", fontsize=9,
                 bbox=dict(facecolor="black", alpha=0.4, pad=1))

plt.tight_layout()
plt.show()
print(f"GT boxes: {len(gt_boxes)}  |  Predicted boxes: {len(pred_boxes)}")

---
## Stage 2 — MedSAM Lesion Segmentation

Prepare 1024×1024 npz files, then fine-tune MedSAM (SAM ViT-B) with bbox prompts for binary lesion segmentation.

In [ ]:
# 2.1 Prepare MedSAM dataset
from medsam.prepare_data import get_image_ids, split_ids, prepare_medsam

image_ids = get_image_ids()
train_ids, val_ids = split_ids(image_ids)
print(f"Split: {len(train_ids)} train, {len(val_ids)} val")
prepare_medsam(train_ids, val_ids)

In [ ]:
# 2.2 Train MedSAM
from medsam.train_segmentation import train as train_medsam

train_medsam(epochs=50, batch=4, lr=1e-4)

---
## Stage 3 — Attribute Segmentation

Prepare attribute dataset (5 dermoscopic attributes), then train the SAM encoder + custom decoder model.

In [ ]:
# 3.1 Prepare attribute dataset
from medsam.prepare_data import get_image_ids, split_ids, prepare_attributes

image_ids = get_image_ids()
train_ids, val_ids = split_ids(image_ids)
prepare_attributes(train_ids, val_ids)

In [ ]:
# 3.2 Train attribute segmentation
from medsam.train_attributes import train as train_attr

train_attr(epochs=80, batch=4, lr=1e-4)

---
## Save Results

Zip trained weights and prepare for download via the Kaggle **Output** tab.

In [ ]:
import zipfile
from pathlib import Path

output_dir = Path("/kaggle/working/outputs")
output_zip = "/kaggle/working/trained_weights.zip"

if output_dir.exists():
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in output_dir.rglob("*"):
            if f.is_file():
                arcname = f.relative_to("/kaggle/working")
                zf.write(f, arcname)
                print(f"  Added: {arcname}")
    print(f"\nSaved to {output_zip}")
    print("Go to the Kaggle Output tab to download trained_weights.zip")
else:
    print("No outputs directory found — training may not have completed successfully.")